# EUW Mean-Data Periodograms vs Mean Player Periodograms

            This notebook compares two different summaries:

            - periodogram of the hourly mean score, after averaging games within each hour
            - mean of the individual player periodograms, after computing one periodogram per player

            These are not equivalent operations. The comparison shows whether a
            pooled rhythm survives averaging across players, and whether player
            rhythms are coherent in phase.

## Setup

            In Colab, the notebook mounts Google Drive and looks for the raw Riot
            Parquet at the same shared-drive path used by the other release
            notebooks: `/content/drive/Shareddrives/MSc_2026_Riot/db/riotData.parquet`.

            The notebook also needs the repository Python files. If they are not
            already present in the runtime or Drive, the setup cell tries to clone
            the release repository into `/content/MSc2026_LoL_Release`.

In [ ]:
# Local users should normally use the uv environment from README.md.
# This cell only installs missing packages when the notebook is opened in Colab.
import importlib.util
import subprocess
import sys

MODULE_TO_PACKAGE = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "duckdb": "duckdb",
    "astropy": "astropy",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "joblib": "joblib",
}

missing = [
    package
    for module, package in MODULE_TO_PACKAGE.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("Notebook packages are available.")

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore[import-not-found]  # noqa: F401
    except ImportError:
        return False
    return True


IN_COLAB = running_in_colab()
if IN_COLAB:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")


# Override this if your repository folder has a different Colab/Drive location.
ROOT_OVERRIDE = None
REPO_URL = "https://github.com/wadelab/MSc2026_LoL_Release.git"


def find_repo_root() -> Path | None:
    if ROOT_OVERRIDE is not None:
        candidate = Path(ROOT_OVERRIDE).expanduser()
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
        raise FileNotFoundError(f"ROOT_OVERRIDE does not contain riot_analysis.py: {candidate}")

    candidates = list(Path.cwd().resolve().parents)
    candidates.insert(0, Path.cwd().resolve())
    candidates.extend(
        [
            Path("/content/MSc2026_LoL_Release"),
            Path("/content/drive/MyDrive/MSc2026_LoL_Release"),
            Path("/content/drive/Shareddrives/MSc_2026_Riot/MSc2026_LoL_Release"),
        ]
    )
    for candidate in candidates:
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
    return None


ROOT = find_repo_root()
if ROOT is None and IN_COLAB:
    clone_target = Path("/content/MSc2026_LoL_Release")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)], check=True)
    ROOT = find_repo_root()

if ROOT is None:
    raise FileNotFoundError(
        "Could not find riot_analysis.py. Set ROOT_OVERRIDE to the repository folder."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 120

print(f"Repository root: {ROOT}")
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
from riot_analysis import (
                AnalysisConfig,
                GOOD_PCA_COLS,
                add_time_normalized_features,
                average_player_periodograms,
                compute_pca,
                configure_plot_style,
                connect_analysis_database,
                filter_hourly_window,
                filter_metric_outliers,
                load_hourly_metrics,
                load_top_players,
                lomb_scargle_summary,
                period_grid,
                project_player_pca,
                style_axes,
            )

            configure_plot_style()

            PLATFORM = "EUW1"
            TOP_N_PLAYERS = 250
            DB_FILE = ROOT / "riot_local.duckdb"
            PARQUET_FILE = None

            config = AnalysisConfig(
                platform=PLATFORM,
                top_n_players=TOP_N_PLAYERS,
                max_hour_limit=5000,
                min_period_h=6,
                max_period_h=48,
                player_period_step_h=0.5,
                n_jobs=2 if IN_COLAB else 8,
                output_root=ROOT / "results",
            )
            conn = connect_analysis_database(DB_FILE, parquet_file=PARQUET_FILE)

## Build the hourly PCA basis, then project player games

            We fit PCA on hourly aggregate rates, then project individual player
            games into that same PCA space. This keeps PC1 and PC2 comparable to
            the rest of the workflow.

In [ ]:
hourly_metrics = load_hourly_metrics(conn, PLATFORM)
            hourly_metrics = filter_hourly_window(hourly_metrics, config.max_hour_limit)
            hourly_metrics, numeric_cols = add_time_normalized_features(hourly_metrics)
            hourly_metrics = filter_metric_outliers(hourly_metrics, numeric_cols)
            pca = compute_pca(hourly_metrics, numeric_cols, GOOD_PCA_COLS)

            top_players, player_data = load_top_players(conn, PLATFORM, TOP_N_PLAYERS)
            player_data = project_player_pca(player_data, numeric_cols, pca)
            player_data["TIMESTAMP"] = pd.to_numeric(player_data["TIMESTAMP"], errors="coerce")
            player_data["hour_idx"] = np.floor(player_data["TIMESTAMP"] / 3600000.0)

            metric_map = {
                "PC1": "perf_factor_pc1",
                "PC2": "perf_factor_pc2",
                "DeltaMMR": "delta_mmr",
            }

            print(f"Loaded {len(top_players):,} players and {len(player_data):,} game rows")
            display(top_players.head(10))

## Compute the two periodogram summaries

            The mean-data periodogram is computed after grouping the same selected
            players into hourly means. The individual summary first computes one
            periodogram per player and then averages the powers.

In [ ]:
frequency, period = period_grid(config, player=True)


            def aggregate_periodogram(data: pd.DataFrame, score_col: str) -> dict | None:
                grouped = (
                    data.dropna(subset=["hour_idx", score_col])
                    .groupby("hour_idx", as_index=False)[score_col]
                    .mean()
                    .sort_values("hour_idx")
                )
                if len(grouped) <= 10 or grouped[score_col].std() == 0:
                    return None
                result = lomb_scargle_summary(
                    grouped["hour_idx"].to_numpy(dtype=float),
                    grouped[score_col].to_numpy(dtype=float),
                    frequency,
                    period,
                )
                result["n_hour_bins"] = len(grouped)
                return result


            mean_data_periodograms = {
                label: aggregate_periodogram(player_data, score_col)
                for label, score_col in metric_map.items()
            }

            individual_periodograms = average_player_periodograms(player_data, metric_map, config)

            rows = []
            for label in metric_map:
                mean_result = mean_data_periodograms[label]
                individual_result = individual_periodograms[label]
                rows.append(
                    {
                        "metric": label,
                        "mean_data_hour_bins": None if mean_result is None else mean_result["n_hour_bins"],
                        "mean_data_best_period_h": None if mean_result is None else mean_result["best_period"],
                        "individual_valid_players": individual_result["valid_players"],
                        "individual_mean_best_period_h": individual_result["best_period"],
                    }
                )

            comparison_summary = pd.DataFrame(rows)
            display(comparison_summary)

## Plot normalized curves and their difference

            Lomb-Scargle power scales with the variance of the series being
            analyzed, so the comparison below normalizes each curve by its own
            maximum. The difference plot is therefore about shape and peak
            location, not absolute power.

In [ ]:
def normalize_power(power):
                power = np.asarray(power, dtype=float)
                max_power = np.nanmax(power)
                if not np.isfinite(max_power) or max_power <= 0:
                    return np.full_like(power, np.nan)
                return power / max_power


            fig, axes = plt.subplots(len(metric_map), 2, figsize=(15, 4.5 * len(metric_map)), sharex=True)
            axes = np.atleast_2d(axes)
            difference_rows = []

            for row, label in enumerate(metric_map):
                mean_result = mean_data_periodograms[label]
                individual_result = individual_periodograms[label]
                ax_curve = axes[row, 0]
                ax_diff = axes[row, 1]

                if mean_result is None or individual_result["mean_power"] is None:
                    ax_curve.set_title(f"{label}: missing periodogram")
                    ax_diff.set_title(f"{label}: missing difference")
                    continue

                mean_scaled = normalize_power(mean_result["power"])
                individual_scaled = normalize_power(individual_result["mean_power"])
                difference = mean_scaled - individual_scaled
                largest_gap_idx = int(np.nanargmax(np.abs(difference)))

                ax_curve.plot(period, mean_scaled, color="#1d3557", linewidth=2.2, label="Periodogram of hourly mean")
                ax_curve.plot(period, individual_scaled, color="#e76f51", linewidth=2.0, label="Mean of player periodograms")
                ax_curve.axvline(24.0, color="#8d99ae", linestyle="--", linewidth=1.1)
                ax_curve.set_title(f"{label}: normalized periodogram summaries")
                ax_curve.set_ylabel("Power / max power")
                ax_curve.legend(frameon=False)
                style_axes(ax_curve)

                ax_diff.plot(period, difference, color="#2a9d8f", linewidth=2.0)
                ax_diff.axhline(0.0, color="#8d99ae", linestyle="--", linewidth=1.0)
                ax_diff.axvline(period[largest_gap_idx], color="#264653", linestyle=":", linewidth=1.2)
                ax_diff.set_title(f"{label}: hourly-mean minus mean-player power")
                ax_diff.set_ylabel("Normalized difference")
                style_axes(ax_diff)

                difference_rows.append(
                    {
                        "metric": label,
                        "largest_absolute_difference_period_h": period[largest_gap_idx],
                        "largest_absolute_difference": difference[largest_gap_idx],
                        "difference_at_24h": difference[int(np.argmin(np.abs(period - 24.0)))],
                    }
                )

            for ax in axes[-1, :]:
                ax.set_xlabel("Period (hours)")

            fig.tight_layout()
            plt.show()

            display(pd.DataFrame(difference_rows))
            conn.close()